In [1]:
import sqlite3
import pandas as pd

df= pd.read_csv("cleaned_customer_shopping_behaviour.csv")
print("Data loaded Sucessfully!!")

Data loaded Sucessfully!!


In [2]:
conn = sqlite3.connect("cleaned_customer_shopping_behaviour.db")
df.to_sql(
    "cleaned_customer_shopping_behaviour",
    conn,
    if_exists="replace",
    index=False
)

print("Data imported into SQLite successfully!")

Data imported into SQLite successfully!


In [3]:
query = """
SELECT *
FROM cleaned_customer_shopping_behaviour
LIMIT 10;
"""

pd.read_sql_query(query, conn)

,customer_id,age,gender,item_purchased,category,purchased_amount,location,size,color,season,review_rating,subscription_status,shipping_type,discount_applied,previous_purchases,payment_method,frequency_of_purchases,review__rating,age_group,purchased_frequency_days
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,14,Venmo,Fortnightly,3.1,Middle_Age,14
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,2,Cash,Fortnightly,3.1,Young,14
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,23,Credit Card,Weekly,3.1,Middle_Age,7
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,49,PayPal,Weekly,3.5,Young,7
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,31,PayPal,Annually,2.7,Middle_Age,365
5,6,46,Male,Sneakers,Footwear,20,Wyoming,M,White,Summer,2.9,Yes,Standard,Yes,14,Venmo,Weekly,2.9,Middle_Age,7
6,7,63,Male,Shirt,Clothing,85,Montana,M,Gray,Fall,3.2,Yes,Free Shipping,Yes,49,Cash,Quarterly,3.2,Senior,90
7,8,27,Male,Shorts,Clothing,34,Louisiana,L,Charcoal,Winter,3.2,Yes,Free Shipping,Yes,19,Credit Card,Weekly,3.2,Young,7
8,9,26,Male,Coat,Outerwear,97,West Virginia,L,Silver,Summer,2.6,Yes,Express,Yes,8,Venmo,Annually,2.6,Young,365
9,10,57,Male,Handbag,Accessories,31,Missouri,M,Pink,Spring,4.8,Yes,2-Day Shipping,Yes,4,Cash,Quarterly,4.8,Middle_Age,90


In [4]:
#--Q1. What is the total revenue generated by male vs. female customers?

query = """
select gender, SUM (purchased_amount) as revnue 
from cleaned_customer_shopping_behaviour
group by gender;
"""

pd.read_sql_query(query, conn)

,gender,revnue
0,Female,75191
1,Male,157890


In [10]:
#--Q2. Which customers used a discount but still spent more than the average purchase amount? 

query = """
select customer_id , purchased_amount 
from cleaned_customer_shopping_behaviour
where discount_applied = 'Yes' AND  purchased_amount >= (select AVG(purchased_amount) from cleaned_customer_shopping_behaviour);
"""

pd.read_sql_query(query, conn)

,customer_id,purchased_amount
0,2,64
1,3,73
2,4,90
3,7,85
4,9,97
...,...,...
834,1667,64
835,1671,73
836,1673,73
837,1674,62


In [18]:
-- Q3. Which are the top 5 products with the highest average review rating?

query= """
select item_purchased , round(AVG(review_rating),2) as "Average Product Rating"
from cleaned_customer_shopping_behaviour
group by item_purchased
order by (AVG(review_rating)) desc
LIMIT 5;
"""
pd.read_sql_query(query, conn)

Object `rating` not found.


,item_purchased,Average Product Rating
0,Gloves,3.86
1,Sandals,3.84
2,Boots,3.82
3,Hat,3.80
4,Skirt,3.78


In [19]:
#--Q4. Compare the average Purchase Amounts between Standard and Express Shipping

query = """
select shipping_type , round(AVG(purchased_amount),2) 
from cleaned_customer_shopping_behaviour
where shipping_type in ('Standard','Express')
group by shipping_type;
"""
pd.read_sql_query(query, conn)

,shipping_type,"round(AVG(purchased_amount),2)"
0,Express,60.48
1,Standard,58.46


In [20]:
#--Q5. What is the revenue contribution of each age group?

query = """
select age_group, sum(purchased_amount) as revenue 
from cleaned_customer_shopping_behaviour
group by age_group
order by revenue desc;
"""
pd.read_sql_query(query , conn)


,age_group,revenue
0,Young,62143
1,Middle_Age,59197
2,Young_Adult,55978
3,Senior,55763


In [23]:
#--Q6. Do subscribed customers spend more? Compare average spend and total revenue 
 #between subscribers and non-subscribers.

query = """
select subscription_status , 
count(customer_id) as total_customers,
round(AVG(purchased_amount),2) as average_spend ,
round(SUM(purchased_amount),2) as total_revenue
from cleaned_customer_shopping_behaviour
group by subscription_status 
order by average_spend,total_revenue;
"""
pd.read_sql_query(query , conn)

,subscription_status,total_customers,average_spend,total_revenue
0,Yes,1053,59.49,62645.0
1,No,2847,59.87,170436.0


In [26]:
#--Q7. Which 5 products have the highest percentage of purchases with discounts applied?

query = """
select item_purchased , 
 ROUND(100.0 * SUM(CASE WHEN discount_applied = 'Yes' THEN 1 ELSE 0 END)/COUNT(*),2) AS discount_rate
FROM cleaned_customer_shopping_behaviour
GROUP BY item_purchased
ORDER BY discount_rate DESC
LIMIT 5;
"""
pd.read_sql_query(query , conn)

,item_purchased,discount_rate
0,Hat,50.00
1,Sneakers,49.66
2,Coat,49.07
3,Sweater,48.17
4,Pants,47.37


In [28]:
#--Q8. Segment customers into New, Returning, and Loyal based on their total 
#-- number of previous purchases, and show the count of each segment. 

query = """
with customer_type as (
SELECT customer_id, previous_purchases,
CASE 
    WHEN previous_purchases = 1 THEN 'New'
    WHEN previous_purchases BETWEEN 2 AND 10 THEN 'Returning'
    ELSE 'Loyal'
    END AS customer_segment
FROM cleaned_customer_shopping_behaviour)

select customer_segment,count(*) AS "Number of Customers" 
from customer_type 
group by customer_segment;
"""
pd.read_sql_query(query , conn)


,customer_segment,Number of Customers
0,Loyal,3116
1,New,83
2,Returning,701


In [30]:
#--Q9. What are the top 3 most purchased products within each category? 

query = """
WITH item_counts AS (
    SELECT category,
           item_purchased,
           COUNT(customer_id) AS total_orders,
           ROW_NUMBER() OVER (PARTITION BY category ORDER BY COUNT(customer_id) DESC) AS item_rank
    FROM cleaned_customer_shopping_behaviour
    GROUP BY category, item_purchased
)
SELECT item_rank,category, item_purchased, total_orders
FROM item_counts
WHERE item_rank <=3;
"""
pd.read_sql_query(query, conn)

,item_rank,category,item_purchased,total_orders
0,1,Accessories,Jewelry,171
1,2,Accessories,Sunglasses,161
2,3,Accessories,Belt,161
3,1,Clothing,Pants,171
4,2,Clothing,Blouse,171
5,3,Clothing,Shirt,169
6,1,Footwear,Sandals,160
7,2,Footwear,Shoes,150
8,3,Footwear,Sneakers,145
9,1,Outerwear,Jacket,163


In [32]:
#--Q10. Are customers who are repeat buyers (more than 5 previous purchases) also likely to subscribe?

query = """
select subscription_status,
count(customer_id) as repeat_customer
from cleaned_customer_shopping_behaviour
where previous_purchases>5
group by subscription_status;
"""
pd.read_sql_query(query , conn)

,subscription_status,repeat_customer
0,No,2518
1,Yes,958
